# Conditional Flow Matching + DiT, jointly trained on two real geometries

See README.md

In [13]:
# from google.colab import drive
# drive.mount('/content/drive/')
# %cd /content/drive/MyDrive/poselab/cern/cfm-dit_voxel/

In [14]:
%pip install --upgrade pip -qqq
%pip install scipy plotly torch torchvision h5py numpy tqdm wandb -qqq

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [15]:
import math
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path
import einops
import datetime
import sys
sys.path.append("../")
from utilities import *
import wandb


torch.cuda.empty_cache()

In [16]:
CFG = dict(
    # --- data: two real geometries jointly trained ---
    # geom_id=0: CaloChallenge Dataset 2 (SiW), fixed incidence (theta=pi/2, phi=0)
    # geom_id=1: LEMURS FCCeeALLEGRO, variable incidence (phi, theta) per sample
    calo_train_data_path = "../../calo-data/dataset_2_1.hdf5",
    calo_test_data_path  = "../../calo-data/dataset_2_2.hdf5",
    lemurs_data_path     = "../../calo-data/FCCeeALLEGRO/LEMURS_FCCeeALLEGRO_gamma_100kEvents_1GeV100GeV_GPSflat_part1.h5",
    lemurs_test_dir      = "../../calo-data/FCCeeALLEGRO/testing",  # fixed-(E,phi,theta) eval files
    showers_key         = "showers",           # shared key name, both files
    lemurs_energies_key = "incident_energy",   # LEMURS key (CaloChallenge hardcodes "incident_energies")
    lemurs_phi_key      = "incident_phi",
    lemurs_theta_key    = "incident_theta",
    voxel_shape = (45, 16, 9),
    n_samples   = None,

    # --- model (CaloArt-inspired DiT backbone) ---
    patch_size   = (5, 4, 3),   # divides (45,16,9) -> grid (9,4,3) = 108 tokens
    embed_dim    = 384,
    depth        = 8,
    num_heads    = 8,           # head_dim=48, divisible by 6 for 3D axial RoPE (16 dims/axis)
    mlp_ratio    = 4,
    cond_dim     = 128,
    n_geometries = 3,           # 0=SiW (CaloChallenge Ds2), 1=FCCeeALLEGRO, 2=reserved for a future geometry
    dropout      = 0.0,
    compile      = False,       # set True on PyTorch 2+ for extra kernel-fusion speedup

    # --- conditional flow matching ---
    sampling_steps  = 50,
    sampling_method = "heun",   # "euler" (1 NFE/step) or "heun" (2 NFE/step, more accurate)

    # --- training ---
    epochs    = 200,
    batch_size= 64,
    lr        = 1e-3,
    grad_clip = 1.0,
    device    = "cuda" if torch.cuda.is_available() else "cpu",
    log_every = 10,

    # --- checkpointing ---
    ckpt_dir  = "checkpoints",
    save_every= 100,
    scratch   = 1,

    # --- wandb logging ---
    wandb_log    = 0,
    wandb_project= "cfm-dit_voxel-multigeo",
    wandb_entity = "pgeorgantopoulos-ntua",
)

In [17]:
if CFG["wandb_log"]:
    wandb.init(
        entity=CFG["wandb_entity"],
        project=CFG["wandb_project"],
        config = CFG
    )

## Dataset

In [18]:
import sys
sys.path.append("../CaloChallenge/code")
from HighLevelFeatures import HighLevelFeatures

hlf = HighLevelFeatures('electron', filename='../CaloChallenge/code/binning_dataset_2.xml')
# Two real geometries, jointly trained (see ../utilities.py:multi_geometry_dataloaders):
#   geom_id=0 -> CaloChallenge Dataset 2 (SiW), fixed incidence
#   geom_id=1 -> LEMURS FCCeeALLEGRO, variable (phi, theta) incidence
# Each batch: (x, cond, phi, theta, geom_id). inverses[geom_id] maps [-1,1] -> raw MeV
# for that geometry (each has its own fitted log-normalisation min/max).
train_loader, test_loader, inverses = multi_geometry_dataloaders(CFG, hlf)

Features in dataset_2_1.hdf5: ['incident_energies', 'showers']


Showers shape: (80000, 6480)
Incident energies shape: (80000, 1)
showers_4d shape: (80000, 9, 16, 45)
Features in dataset_2_1.hdf5: ['incident_energies', 'showers']
Showers shape: (20000, 6480)
Incident energies shape: (20000, 1)
showers_4d shape: (20000, 9, 16, 45)
Train: 80000 | Test: 20000
Train: 79953 | Test: 19989 | Voxel shape: (9, 16, 45)


## Model

In [19]:
def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """Sinusoidal embedding (Vaswani et al.); works for continuous t as well as integer steps."""
    half  = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(half, device=t.device) / (half - 1)
    )
    args = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)  # (B, dim)


class TimeEmbedding(nn.Module):
    """Embeds the continuous flow-matching time t (scaled by 1000 for frequency spread)."""

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim * 4),
        )

    def forward(self, t):
        return self.net(sinusoidal_embedding(t, self.dim))  # (B, dim*4)


class AngleEmbedding(nn.Module):
    """Embeds incident (phi, theta) via sin/cos features -> MLP (avoids the phi=0/2pi
    discontinuity a raw-radian input would have). Used identically for both geometries;
    CaloChallenge Dataset 2 always supplies the fixed (phi=0, theta=pi/2)."""

    def __init__(self, embed_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, embed_dim), nn.SiLU(), nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, phi: torch.Tensor, theta: torch.Tensor):
        feats = torch.stack([phi.sin(), phi.cos(), theta.sin(), theta.cos()], dim=-1)  # (B, 4)
        return self.net(feats)


# ── Patchify / unpatchify ──────────────────────────────────────────────────────

class PatchEmbed3D(nn.Module):
    """Non-overlapping 3D patch embedding (large patches, CaloArt-style)."""

    def __init__(self, in_channels: int, patch_size: tuple, embed_dim: int):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv3d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor):
        x = self.proj(x)                       # (B, embed_dim, D_p, H_p, W_p)
        grid_shape = x.shape[2:]
        x = einops.rearrange(x, 'b c d h w -> b (d h w) c')
        return x, grid_shape


class FinalLayer3D(nn.Module):
    """adaLN-Zero output head: projects tokens back to patches and unpatchifies to the grid."""

    def __init__(self, embed_dim: int, patch_size: tuple, out_channels: int = 1):
        super().__init__()
        self.patch_size = patch_size
        self.out_channels = out_channels
        self.norm = nn.LayerNorm(embed_dim, elementwise_affine=False)
        self.adaLN = nn.Sequential(nn.SiLU(), nn.Linear(embed_dim, embed_dim * 2))
        nn.init.zeros_(self.adaLN[-1].weight)
        nn.init.zeros_(self.adaLN[-1].bias)
        patch_vol = patch_size[0] * patch_size[1] * patch_size[2]
        self.out_proj = nn.Linear(embed_dim, patch_vol * out_channels)
        nn.init.zeros_(self.out_proj.weight)   # zero-init -> v_theta(x,0)=0 at init
        nn.init.zeros_(self.out_proj.bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor, grid_shape: tuple):
        shift, scale = self.adaLN(cond).chunk(2, dim=-1)
        x = self.norm(x) * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
        x = self.out_proj(x)                   # (B, N, patch_vol*out_channels)
        d_p, h_p, w_p = grid_shape
        pd, ph, pw = self.patch_size
        x = einops.rearrange(
            x, 'b (d h w) (pd ph pw c) -> b c (d pd) (h ph) (w pw)',
            d=d_p, h=h_p, w=w_p, pd=pd, ph=ph, pw=pw, c=self.out_channels,
        )
        return x


# ── 3D axial RoPE (CaloArt-inspired) ───────────────────────────────────────────

class AxialRoPE3D(nn.Module):
    """
    3D axial rotary position embedding.

    Splits each attention head's dimension into 3 equal chunks — one per patch-grid
    axis (depth/layer, angle, radius) — and applies standard (NeoX-style) RoPE
    rotation to each chunk using that axis's *patch-grid* coordinate as the rotary
    position. This is the raw-grid-position version described for CaloArt. Even
    though this model already trains jointly on two geometries (SiW + FCCeeALLEGRO),
    both share the same (45,16,9) voxel grid, so raw patch-grid position is still a
    consistent coordinate system across them. Making this transfer across detectors
    with genuinely different binning/grid shapes would mean computing the per-axis
    position from physical units (X0, Rm) instead of the patch index — not yet
    needed here, but a drop-in swap to this one class if a third, differently-binned
    geometry is ever added (see top-level README.md, "Architectural refinements"
    section).
    """

    def __init__(self, head_dim: int, grid_shape: tuple, base: float = 10000.0):
        super().__init__()
        assert head_dim % 6 == 0, "head_dim must be divisible by 6 for 3D axial RoPE (2 * 3 axes)"
        self.axis_dim = head_dim // 3
        self.grid_shape = grid_shape

        inv_freq = 1.0 / (base ** (torch.arange(0, self.axis_dim, 2).float() / self.axis_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        d_p, h_p, w_p = grid_shape
        coords = torch.stack(torch.meshgrid(
            torch.arange(d_p), torch.arange(h_p), torch.arange(w_p), indexing="ij"
        ), dim=-1).reshape(-1, 3).float()      # (N, 3) — per-token (d, h, w) patch coord
        self.register_buffer("coords", coords, persistent=False)

    def _freqs_for_axis(self, pos):
        freqs = pos[:, None] * self.inv_freq[None, :]   # (N, axis_dim//2)
        return torch.cat([freqs, freqs], dim=-1)         # (N, axis_dim)

    def forward(self, q: torch.Tensor, k: torch.Tensor):
        # q, k: (B, heads, N, head_dim)
        def apply(x):
            chunks = x.split(self.axis_dim, dim=-1)     # one chunk per axis
            out = []
            for axis, xc in enumerate(chunks):
                pos = self.coords[:, axis].to(x.device)
                freqs = self._freqs_for_axis(pos)
                cos, sin = freqs.cos()[None, None], freqs.sin()[None, None]
                x1, x2 = xc.chunk(2, dim=-1)
                rot = torch.cat([-x2, x1], dim=-1)
                out.append(xc * cos + rot * sin)
            return torch.cat(out, dim=-1)
        return apply(q), apply(k)


class Attention3DRoPE(nn.Module):
    """Multi-head self-attention over patch tokens with 3D axial RoPE on Q/K."""

    def __init__(self, embed_dim: int, num_heads: int, grid_shape: tuple, dropout: float = 0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.rope = AxialRoPE3D(self.head_dim, grid_shape)
        self.dropout = dropout

    def forward(self, x: torch.Tensor):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                # (B, heads, N, head_dim)
        q, k = self.rope(q, k)
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0)
        out = out.transpose(1, 2).reshape(B, N, C)
        return self.proj(out)


# ── DiT block (adaLN-Zero conditioning, Peebles & Xie 2023) ────────────────────

class DiTBlock3D(nn.Module):
    """Pre-LN transformer block with adaLN-Zero modulation from the (t, E, geometry) embedding."""

    def __init__(self, embed_dim: int, num_heads: int, grid_shape: tuple,
                 mlp_ratio: int = 4, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim, elementwise_affine=False)
        self.attn  = Attention3DRoPE(embed_dim, num_heads, grid_shape, dropout)
        self.norm2 = nn.LayerNorm(embed_dim, elementwise_affine=False)
        hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden), nn.GELU(approximate="tanh"),
            nn.Dropout(dropout), nn.Linear(hidden, embed_dim), nn.Dropout(dropout),
        )
        # 6 = (shift, scale, gate) x (attn, mlp)
        self.adaLN = nn.Sequential(nn.SiLU(), nn.Linear(embed_dim, embed_dim * 6))
        nn.init.zeros_(self.adaLN[-1].weight)   # zero-init -> block is identity at init
        nn.init.zeros_(self.adaLN[-1].bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor):
        shift1, scale1, gate1, shift2, scale2, gate2 = self.adaLN(cond).chunk(6, dim=-1)
        h = self.norm1(x) * (1 + scale1.unsqueeze(1)) + shift1.unsqueeze(1)
        x = x + gate1.unsqueeze(1) * self.attn(h)
        h = self.norm2(x) * (1 + scale2.unsqueeze(1)) + shift2.unsqueeze(1)
        x = x + gate2.unsqueeze(1) * self.mlp(h)
        return x


# ── Full backbone ───────────────────────────────────────────────────────────────

class ConditionalFlowMatchingDiT(nn.Module):
    """
    CaloArt-inspired Diffusion Transformer, trained as a conditional flow-matching
    velocity predictor. Pure transformer — no convolutional U-Net path.

    Conditioning: adaLN-Zero on every block from a merged embedding of
      - flow-matching time t (continuous, in [0, 1])
      - incident energy log10(E_inc / GeV)
      - incident angle (phi, theta), via sin/cos features — real per-sample values for
        LEMURS FCCeeALLEGRO, a fixed (phi=0, theta=pi/2) constant for CaloChallenge
        Dataset 2 (perpendicular incidence, azimuthally symmetric)
      - a geometry embedding (n_geometries=3: 0=SiW/CaloChallenge Dataset 2,
        1=FCCeeALLEGRO, 2=reserved for a future third geometry), matching
        CaloDiT-2's one-hot-embedding conditioning pattern

    Args
        x_t     : (B, 1, D, H, W)  interpolated shower at flow-time t
        t       : (B,)              continuous flow time in [0, 1]
        cond    : (B,)              log10(E_inc / GeV)
        phi, theta : (B,)           incident angle in radians
        geom_id : (B,) long or None — geometry index; defaults to all-zeros

    Returns
        v_pred : (B, 1, D, H, W)  predicted velocity field x1 - x0
    """

    def __init__(self, voxel_shape, patch_size=(5, 4, 3), embed_dim=384, depth=8,
                 num_heads=8, mlp_ratio=4, cond_dim=128, n_geometries=3, dropout=0.0):
        super().__init__()
        self.voxel_shape = voxel_shape
        self.patch_embed = PatchEmbed3D(1, patch_size, embed_dim)
        grid_shape = tuple(v // p for v, p in zip(voxel_shape, patch_size))

        self.time_emb    = TimeEmbedding(embed_dim // 4)     # dim*4 == embed_dim
        self.energy_proj = nn.Sequential(
            nn.Linear(1, cond_dim), nn.SiLU(), nn.Linear(cond_dim, embed_dim)
        )
        self.angle_emb   = AngleEmbedding(embed_dim)
        self.geom_embed  = nn.Embedding(n_geometries, embed_dim)
        self.cond_merge  = nn.Sequential(
            nn.Linear(embed_dim * 4, embed_dim), nn.SiLU(), nn.Linear(embed_dim, embed_dim)
        )

        self.blocks = nn.ModuleList([
            DiTBlock3D(embed_dim, num_heads, grid_shape, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.final = FinalLayer3D(embed_dim, patch_size, out_channels=1)

    def forward(self, x_t, t, cond, phi, theta, geom_id=None):
        if geom_id is None:
            geom_id = torch.zeros(x_t.shape[0], dtype=torch.long, device=x_t.device)

        tokens, grid_shape = self.patch_embed(x_t)

        t_emb = self.time_emb(t * 1000.0)                # (B, embed_dim)
        e_emb = self.energy_proj(cond.unsqueeze(-1))      # (B, embed_dim)
        a_emb = self.angle_emb(phi, theta)                # (B, embed_dim)
        g_emb = self.geom_embed(geom_id)                  # (B, embed_dim)
        c     = self.cond_merge(torch.cat([t_emb, e_emb, a_emb, g_emb], dim=-1))  # (B, embed_dim)

        for block in self.blocks:
            tokens = block(tokens, c)

        return self.final(tokens, c, grid_shape)


class RectifiedFlow:
    """
    Conditional Flow Matching via the rectified-flow linear path
    (Liu et al., 2022, arXiv:2209.03003; Lipman et al., 2023 CFM with sigma_min=0).

    Training: sample t~U(0,1) and x0~N(0,I), interpolate x_t = (1-t)*x0 + t*x1,
              regress the (t-independent along the path) target velocity u_t = x1 - x0.
    Sampling: integrate the ODE dx/dt = v_theta(x, t, cond, phi, theta, geom_id) from
              t=0 to t=1, via Euler (1 NFE/step) or Heun/RK2 (2 NFE/step, better
              quality at low steps).
    """

    def __init__(self, device: str = "cpu"):
        self.device = device

    def training_loss(self, model, x1, cond, phi, theta, geom_id=None):
        """Sample random t, interpolate x0->x1, return MSE(v_pred, x1 - x0)."""
        B    = x1.shape[0]
        t    = torch.rand(B, device=x1.device)
        x0   = torch.randn_like(x1)
        t_   = t[:, None, None, None, None]
        x_t  = (1 - t_) * x0 + t_ * x1
        target = x1 - x0
        v_pred = model(x_t, t, cond, phi, theta, geom_id)
        return F.mse_loss(v_pred, target)

    @torch.no_grad()
    def sample(self, model, shape, cond, phi, theta, geom_id=None, steps: int = 50, method: str = "heun"):
        """Integrate dx/dt = v_theta from t=0 (noise) to t=1 (data)."""
        device = self.device
        x  = torch.randn(shape, device=device)
        ts = torch.linspace(0.0, 1.0, steps + 1, device=device)

        for i in range(steps):
            t_cur, t_next = ts[i], ts[i + 1]
            dt    = t_next - t_cur
            t_vec = t_cur.expand(shape[0])
            v1    = model(x, t_vec, cond, phi, theta, geom_id)

            if method == "euler":
                x = x + dt * v1
            elif method == "heun":
                x_euler     = x + dt * v1
                t_next_vec  = t_next.expand(shape[0])
                v2          = model(x_euler, t_next_vec, cond, phi, theta, geom_id)
                x           = x + dt * 0.5 * (v1 + v2)
            else:
                raise ValueError(f"Unknown method: {method}")

        return x

## Train

In [20]:
def train(cfg: dict, train_loader):
    device    = cfg["device"]
    ckpt_dir  = Path(cfg["ckpt_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    model = ConditionalFlowMatchingDiT(
        voxel_shape  = cfg["voxel_shape"],
        patch_size   = cfg["patch_size"],
        embed_dim    = cfg["embed_dim"],
        depth        = cfg["depth"],
        num_heads    = cfg["num_heads"],
        mlp_ratio    = cfg["mlp_ratio"],
        cond_dim     = cfg["cond_dim"],
        n_geometries = cfg["n_geometries"],
        dropout      = cfg["dropout"],
    ).to(device)

    if cfg.get("compile", False):
        model = torch.compile(model)

    flow = RectifiedFlow(device=device)

    n_params = sum(p.numel() for p in model.parameters())
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    vram_gib = (param_bytes + buffer_bytes) / 1024**3
    print(f"  Parameters : {n_params / 1e6:.2f} M")
    print(f"  VRAM       : {vram_gib:.3f} GiB")

    opt    = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched  = CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    start_epoch = 0
    loss_history = []

    if not cfg['scratch']:
        latest = sorted(ckpt_dir.glob("ckpt_epoch*.pt"))
        if latest:
            state = torch.load(latest[-1], map_location=device)
            model.load_state_dict(state["model"])
            opt.load_state_dict(state["opt"])
            sched.load_state_dict(state["sched"])
            if "scaler" in state:
                scaler.load_state_dict(state["scaler"])
            start_epoch = state["epoch"] + 1
            print(f"Resumed from {latest[-1]}  (epoch {start_epoch})")

    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        epoch_loss = 0.0

        for x1, cond, phi, theta, geom_id in train_loader:
            x1, cond   = x1.to(device), cond.to(device)
            phi, theta = phi.to(device), theta.to(device)
            geom_id    = geom_id.to(device)

            with torch.autocast("cuda", enabled=(device == "cuda")):
                loss = flow.training_loss(model, x1, cond, phi, theta, geom_id)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()

            epoch_loss += loss.item()

        sched.step()
        avg_loss = epoch_loss / len(train_loader)
        loss_history.append(avg_loss)

        if (epoch + 1) % cfg['log_every'] == 0:
            print(f"Epoch {epoch+1:4d}/{cfg['epochs']}  |  loss={avg_loss:.5f}"
                  f"  lr={sched.get_last_lr()[0]:.2e}")
            if wandb.run is not None:
                wandb.log({"train_loss": avg_loss, "learning_rate": sched.get_last_lr()[0]}, step=epoch + 1)

        if (epoch + 1) % cfg["save_every"] == 0:
            ckpt_path = ckpt_dir / f"ckpt{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M')}_epoch{epoch+1:04d}.pt"
            torch.save({
                "epoch":  epoch,
                "model":  model.state_dict(),
                "opt":    opt.state_dict(),
                "sched":  sched.state_dict(),
                "scaler": scaler.state_dict(),
                "cfg":    cfg,
            }, ckpt_path)
            print(f"  -> saved {ckpt_path}")

    return model, flow, loss_history


# ── Local checkpoint / generation helpers ──────────────────────────────────────
# Not added to ../utilities.py: those DDIM equivalents (load_model_from_checkpoint,
# generate) are hard-wired to UNet3D/DDIMScheduler kwargs and would need a parallel
# code path anyway, so this model gets its own small local versions instead.

def load_fm_checkpoint(cfg: dict, model_cls, flow_cls):
    device   = cfg["device"]
    ckpt_dir = Path(cfg["ckpt_dir"])
    ckpts    = sorted(ckpt_dir.glob("ckpt*.pt"))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found in {ckpt_dir}")

    state      = torch.load(ckpts[-1], map_location=device)
    loaded_cfg = state["cfg"]
    print(f"Loading {ckpts[-1]}  (epoch {state['epoch'] + 1})")
    print(loaded_cfg)

    model = model_cls(
        voxel_shape  = loaded_cfg["voxel_shape"],
        patch_size   = loaded_cfg["patch_size"],
        embed_dim    = loaded_cfg["embed_dim"],
        depth        = loaded_cfg["depth"],
        num_heads    = loaded_cfg["num_heads"],
        mlp_ratio    = loaded_cfg["mlp_ratio"],
        cond_dim     = loaded_cfg["cond_dim"],
        n_geometries = loaded_cfg["n_geometries"],
        dropout      = loaded_cfg["dropout"],
    ).to(device)
    model.load_state_dict(state["model"])
    model.eval()

    flow = flow_cls(device=device)

    n_params     = sum(p.numel() for p in model.parameters())
    param_bytes  = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    print(f"  Parameters : {n_params / 1e6:.2f} M  |  VRAM : {(param_bytes + buffer_bytes) / 1024**3:.3f} GiB")
    print(f"Model loaded from epoch {state['epoch'] + 1}.")
    return model, flow


def generate_fm(model, flow, cfg: dict, inverse, n_samples: int = 4,
                 e_inc_gev: float = 10.0, phi: float = 0.0, theta: float = math.pi / 2,
                 geom_id: int = 0, steps: int = None, method: str = None):
    """Generate shower samples conditioned on (energy, phi, theta, geometry).

    Parameters mirror ../utilities.py's generate(), but integrate the flow-matching
    ODE (steps/method) instead of DDIM's discrete reverse chain (ddim_steps/eta), and
    add angle/geometry conditioning. Defaults (phi=0, theta=pi/2, geom_id=0) match
    CaloChallenge Dataset 2's fixed incidence; pass geom_id=1 + real phi/theta for
    FCCeeALLEGRO, with ``inverse=inverses[1]``.
    """
    model.eval()
    device  = cfg["device"]
    D, H, W = cfg["voxel_shape"]
    cond     = torch.full((n_samples,), math.log10(e_inc_gev * 1e3), device=device)
    phi_t    = torch.full((n_samples,), phi, device=device)
    theta_t  = torch.full((n_samples,), theta, device=device)
    geom_t   = torch.full((n_samples,), geom_id, dtype=torch.long, device=device)
    shape    = (n_samples, 1, D, H, W)
    steps    = cfg["sampling_steps"]  if steps  is None else steps
    method   = cfg["sampling_method"] if method is None else method
    with torch.no_grad():
        samples = flow.sample(model, shape, cond, phi_t, theta_t, geom_t, steps=steps, method=method)
    return inverse(samples.clamp(-1, 1).cpu().numpy())


print("device:", CFG["device"])

device: cuda


In [21]:
model, flow, loss_history_initial = train(CFG, train_loader)

fig = plt.figure(figsize=(10, 6))
plt.plot(loss_history_initial)
plt.title('Training Loss Over Epochs (Initial Run)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
if wandb.run is not None:
    wandb.log({"loss_curve": wandb.Image(fig)})
plt.show()

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 7.66 GiB of which 10.69 MiB is free. Including non-PyTorch memory, this process has 1.38 GiB memory in use. Process 75058 has 6.25 GiB memory in use. Of the allocated memory 1.12 GiB is allocated by PyTorch, and 67.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Load pretrained model

In [ ]:
# No checkpoint exists yet for this model — run the train() cell above first,
# then uncomment this to reload for eval (mirrors ddim-t_voxel.ipynb's pattern).
# model, flow = load_fm_checkpoint(CFG, ConditionalFlowMatchingDiT, RectifiedFlow)

## Eval

### Geometry 0 — CaloChallenge Dataset 2 (SiW): E_INC sweep

In [ ]:
ENERGIES_GEV = [1, 10, 100, 1000, 2000]
N_SWEEP      = 100

with h5py.File(CFG["calo_test_data_path"], "r") as f:
    _all_E       = f["incident_energies"][:].flatten()   # MeV
    _all_showers = f[CFG["showers_key"]][:]

E_INC_MIN = _all_E.min()
E_INC_MAX = _all_E.max()

sweep = {}
for e_gev in ENERGIES_GEV:
    gen = generate_fm(model, flow, CFG, inverse=inverses[0], n_samples=N_SWEEP,
                       e_inc_gev=e_gev, phi=0.0, theta=math.pi / 2, geom_id=0).squeeze(1)  # (N, 45, 16, 9)

    E_mev = e_gev * 1e3
    mask  = (_all_E > E_mev / 2) & (_all_E < E_mev * 2)
    gt    = _all_showers[mask][:N_SWEEP].reshape(-1, *CFG["voxel_shape"])

    sweep[e_gev] = {"gen": gen, "gt": gt}
    if mask.any():
        print(f"E = {e_gev:>5} GeV | gen {gen.shape[0]:>3} | gt {gt.shape[0]:>3}"
            f" | GT range {_all_E[mask].min()/1e3:.2f}–{_all_E[mask].max()/1e3:.2f} GeV")
    else:
        print(f"E = {e_gev:>5} GeV | gen {gen.shape[0]:>3}")

NameError: name 'model' is not defined

### Geometry 0 — Z-profiles

In [ ]:
n_e  = len(ENERGIES_GEV)
fig, axes = plt.subplots(1, n_e, figsize=(5 * n_e, 5), sharey=False)

for ax, e_gev in zip(axes, ENERGIES_GEV):
    gt_z  = sweep[e_gev]["gt"].sum(axis=(2, 3))   # (N, 45) — sum over phi x r
    gen_z = sweep[e_gev]["gen"].sum(axis=(2, 3))

    ax.plot(gt_z.T,  lw=0.35, color="steelblue", alpha=0.5)
    ax.plot(gen_z.T, lw=0.35, color="tomato",    alpha=0.5)
    ax.set_title(f"{e_gev} GeV", fontsize=12)
    ax.set_xlabel("Depth layer (z)")
    ax.grid(linestyle="--", alpha=0.4)
    if ax is axes[0]:
        ax.set_ylabel("Energy [MeV]")

legend_handles = [
    plt.Line2D([], [], color="black", lw=1.5, label="GT"),
    plt.Line2D([], [], color="red",    lw=1.5, label="Generated"),
]
fig.legend(handles=legend_handles, loc="upper right", fontsize=11, framealpha=0.9)
fig.suptitle(f"SiW Z-profiles \u2014 {N_SWEEP} samples per energy", fontsize=13)
fig.tight_layout()
if wandb.run is not None:
    wandb.log({"sweep/siw_z_profiles": wandb.Image(fig)})
plt.show()

### Geometry 0 — Energy Distribution + Radial Profile Comparison

In [ ]:
# plot_comparison expects (K, R, PHI, Z); sweep data is (N, Z, PHI, R) \u2014 transpose needed
def to_rphiz(arr):
    return arr.transpose(0, 3, 2, 1)  # (N, Z, PHI, R) -> (N, R, PHI, Z)

# Only include energy levels that have GT data
available_e = [e for e in ENERGIES_GEV if sweep[e]["gt"].shape[0] > 0]

ref_showers_all  = np.concatenate([to_rphiz(sweep[e]["gt"])  for e in available_e], axis=0)
gen_showers_all  = np.concatenate([to_rphiz(sweep[e]["gen"]) for e in available_e], axis=0)
ref_energies_all = np.concatenate([np.full(sweep[e]["gt"].shape[0],  e * 1e3) for e in available_e])
gen_energies_all = np.concatenate([np.full(sweep[e]["gen"].shape[0], e * 1e3) for e in available_e])

fig = plot_comparison(ref_showers_all, ref_energies_all,
                      gen_showers_all, gen_energies_all,
                      gen_label="CFM+DiT (SiW)", n_cols=len(available_e))
if wandb.run is not None:
    wandb.log({"comparison/siw": wandb.Image(fig)})
plt.show()

### Geometry 1 — LEMURS FCCeeALLEGRO: (E, φ, θ) sweep

In [ ]:
import re
from pathlib import Path

N_SWEEP_LEMURS = 100
_fname_re = re.compile(r"(\d+)GeV_phi([\d.]+)_theta([\d.]+)\.h5$")

lemurs_sweep = {}
for fpath in sorted(Path(CFG["lemurs_test_dir"]).glob("*.h5")):
    m = _fname_re.search(fpath.name)
    e_gev, phi_val, theta_val = float(m.group(1)), float(m.group(2)), float(m.group(3))

    with h5py.File(fpath, "r") as f:
        gt_raw = f[CFG["showers_key"]][:N_SWEEP_LEMURS]     # (N, R, PHI, Z)
    gt = gt_raw.transpose(0, 3, 2, 1)                        # (N, Z, PHI, R) = voxel_shape order

    gen = generate_fm(model, flow, CFG, inverse=inverses[1], n_samples=N_SWEEP_LEMURS,
                       e_inc_gev=e_gev, phi=phi_val, theta=theta_val, geom_id=1).squeeze(1)

    lemurs_sweep[(e_gev, phi_val, theta_val)] = {"gen": gen, "gt": gt}
    print(f"E={e_gev:>4} GeV  phi={phi_val:.2f}  theta={theta_val:.2f}"
          f" | gen {gen.shape[0]:>3} | gt {gt.shape[0]:>3}")

### Geometry 1 — Z-profiles across (E, φ, θ)

In [ ]:
combos = sorted(lemurs_sweep.keys())
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharey=False)

for ax, combo in zip(axes.flat, combos):
    e_gev, phi_val, theta_val = combo
    gt_z  = lemurs_sweep[combo]["gt"].sum(axis=(2, 3))   # (N, 45) — sum over phi x r
    gen_z = lemurs_sweep[combo]["gen"].sum(axis=(2, 3))

    ax.plot(gt_z.T,  lw=0.35, color="steelblue", alpha=0.5)
    ax.plot(gen_z.T, lw=0.35, color="tomato",    alpha=0.5)
    ax.set_title(f"E={e_gev:g} GeV, φ={phi_val:.2f}, θ={theta_val:.2f}", fontsize=10)
    ax.set_xlabel("Depth layer (z)")
    ax.grid(linestyle="--", alpha=0.4)

legend_handles = [
    plt.Line2D([], [], color="black", lw=1.5, label="GT"),
    plt.Line2D([], [], color="red",    lw=1.5, label="Generated"),
]
fig.legend(handles=legend_handles, loc="upper right", fontsize=11, framealpha=0.9)
fig.suptitle(f"FCCeeALLEGRO Z-profiles across (E, φ, θ) — {N_SWEEP_LEMURS} samples/combo", fontsize=13)
fig.tight_layout()
if wandb.run is not None:
    wandb.log({"sweep/allegro_z_profiles": wandb.Image(fig)})
plt.show()

### Geometry 1 — Energy Distribution + Radial Profile Comparison

In [ ]:
# to_rphiz() defined in the Geometry 0 comparison cell above; reused here.
ref_showers_all  = np.concatenate([to_rphiz(lemurs_sweep[c]["gt"])  for c in combos], axis=0)
gen_showers_all  = np.concatenate([to_rphiz(lemurs_sweep[c]["gen"]) for c in combos], axis=0)
ref_energies_all = np.concatenate([np.full(lemurs_sweep[c]["gt"].shape[0],  c[0] * 1e3) for c in combos])
gen_energies_all = np.concatenate([np.full(lemurs_sweep[c]["gen"].shape[0], c[0] * 1e3) for c in combos])

fig = plot_comparison(ref_showers_all, ref_energies_all,
                      gen_showers_all, gen_energies_all,
                      gen_label="CFM+DiT (ALLEGRO)", n_cols=len(combos))
if wandb.run is not None:
    wandb.log({"comparison/allegro": wandb.Image(fig)})
plt.show()